# Order Latency Data
# 订单延迟数据

To obtain more realistic backtesting results, accounting for latencies is crucial. Therefore, it's important to collect both feed data and order data with timestamps to measure your order latency. The best approach is to gather your own order latencies. You can collect order latency based on your live trading or by regularly submitting orders at a price that cannot be filled and then canceling them for recording purposes. However, if you don't have access to them or want to establish a target, you will need to artificially generate order latency. You can model this latency based on factors such as feed latency, trade volume, and the number of events. In this guide, we will demonstrate a simple method to generate order latency from feed latency using a multiplier and offset for adjustment.

为了获得更贴近实际情况的回测结果，考虑延迟因素至关重要。因此，收集带有时间戳的报价数据和订单数据以测量您的订单延迟时间是非常重要的。最佳方法是自行收集订单延迟时间。您可以根据实际交易情况收集订单延迟时间，或者定期以无法成交的价格提交订单，然后取消这些订单以进行记录。然而，如果您无法获取这些数据或者想要设定一个目标，您就需要人为地生成订单延迟时间。您可以根据诸如报价延迟、交易量和事件数量等因素来模拟这种延迟时间。在本指南中，我们将演示一种使用乘数和偏移量从报价延迟时间生成订单延迟时间的简单方法，以进行调整。

First, loads the feed data.

首先，加载数据源信息。

In [1]:
import numpy as np

data = np.load('usdm/ethusdc_20260302.npz')['data']
data

array([(3758096385, 1772409599992000000, 1772409600007726496, 1938.62,  1.152, 0, 0, 0.),
       (3758096385, 1772409599992000000, 1772409600007726496, 1938.66,  0.   , 0, 0, 0.),
       (3489660929, 1772409599992000000, 1772409600007726496, 1939.14,  2.659, 0, 0, 0.),
       ...,
       (3489660929, 1772495999941000000, 1772495999944586742, 2062.97,  0.103, 0, 0, 0.),
       (3489660929, 1772495999941000000, 1772495999944586742, 2062.99, 15.49 , 0, 0, 0.),
       (3489660929, 1772495999941000000, 1772495999944586742, 2229.22,  0.   , 0, 0, 0.)],
      shape=(61971351,), dtype=[('ev', '<u8'), ('exch_ts', '<i8'), ('local_ts', '<i8'), ('px', '<f8'), ('qty', '<f8'), ('order_id', '<u8'), ('ival', '<i8'), ('fval', '<f8')])

For easy manipulation, converts it into a DataFrame.

为了便于操作，将其转换为一个数据框。

In [2]:
import polars as pl

df = pl.DataFrame(data)
df

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
u64,i64,i64,f64,f64,u64,i64,f64
3758096385,1772409599992000000,1772409600007726496,1938.62,1.152,0,0,0.0
3758096385,1772409599992000000,1772409600007726496,1938.66,0.0,0,0,0.0
3489660929,1772409599992000000,1772409600007726496,1939.14,2.659,0,0,0.0
3489660929,1772409599992000000,1772409600007726496,1939.2,0.011,0,0,0.0
3489660929,1772409599992000000,1772409600007726496,1939.27,4.169,0,0,0.0
…,…,…,…,…,…,…,…
3489660929,1772495999941000000,1772495999944586742,2027.09,0.256,0,0,0.0
3489660929,1772495999941000000,1772495999944586742,2027.12,0.15,0,0,0.0
3489660929,1772495999941000000,1772495999944586742,2062.97,0.103,0,0,0.0


Selects only the events that have both a valid exchange timestamp and a valid local timestamp to get feed latency.

仅选取那些同时具有有效交换时间戳和有效本地时间戳的事件，以此来获取数据传输延迟。

In [3]:
from hftbacktest import EXCH_EVENT, LOCAL_EVENT

df = df.filter((pl.col('ev') & EXCH_EVENT == EXCH_EVENT) & (pl.col('ev') & LOCAL_EVENT == LOCAL_EVENT))

Reduces the number of rows by resampling to approximately 1-second intervals.

通过重新采样将数据行的数量减少至大约每 1 秒一个的间隔。

In [4]:
df = df.with_columns(
    pl.col('local_ts').alias('ts')
).group_by_dynamic(
    'ts', every='1000000000i'
).agg(
    pl.col('exch_ts').last(),
    pl.col('local_ts').last()
).drop('ts')

df

exch_ts,local_ts
i64,i64
1772409600950000000,1772409600954881550
1772409601967000000,1772409601989106362
1772409602957000000,1772409602961586180
1772409603972000000,1772409603976628721
1772409604951000000,1772409604955901785
…,…
1772495995985000000,1772495995988617135
1772495996966000000,1772495996986552580
1772495997968000000,1772495997988565647


Converts back to the structured NumPy array.

转换回结构化的 NumPy 数组。

In [5]:
data = df.to_numpy(structured=True)
data

array([(1772409600950000000, 1772409600954881550),
       (1772409601967000000, 1772409601989106362),
       (1772409602957000000, 1772409602961586180), ...,
       (1772495997968000000, 1772495997988565647),
       (1772495998968000000, 1772495998976640649),
       (1772495999941000000, 1772495999944586742)],
      shape=(86400,), dtype=[('exch_ts', '<i8'), ('local_ts', '<i8')])

Generates order latency. Order latency consists of two components: the latency until the order request reaches the exchange's matching engine and the latency until the response arrives backto the localy. Order latency is not the same as feed latency and does not need to be proportional to feed latency. However, for simplicity, we model order latency to be proportional to feed latency using a multiplier and offset.

会产生订单延迟。订单延迟由两个部分组成：订单请求到达交易所匹配引擎的延迟以及响应返回至本地的延迟。订单延迟与数据馈送延迟不同，且不必与数据馈送延迟成正比。然而，为了简便起见，我们使用乘数和偏移量将订单延迟建模为与数据馈送延迟成比例的值。

In [6]:
mul_entry = 4
offset_entry = 0

mul_resp = 3
offset_resp = 0

order_latency = np.zeros(len(data), dtype=[('req_ts', 'i8'), ('exch_ts', 'i8'), ('resp_ts', 'i8'), ('_padding', 'i8')])
for i, (exch_ts, local_ts) in enumerate(data):
    feed_latency = local_ts - exch_ts
    order_entry_latency = mul_entry * feed_latency + offset_entry
    order_resp_latency = mul_resp * feed_latency + offset_resp

    req_ts = local_ts
    order_exch_ts = req_ts + order_entry_latency
    resp_ts = order_exch_ts + order_resp_latency
    
    order_latency[i] = (req_ts, order_exch_ts, resp_ts, 0)
    
order_latency

array([(1772409600954881550, 1772409600974407750, 1772409600989052400, 0),
       (1772409601989106362, 1772409602077531810, 1772409602143850896, 0),
       (1772409602961586180, 1772409602979930900, 1772409602993689440, 0),
       ...,
       (1772495997988565647, 1772495998070828235, 1772495998132525176, 0),
       (1772495998976640649, 1772495999011203245, 1772495999037125192, 0),
       (1772495999944586742, 1772495999958933710, 1772495999969693936, 0)],
      shape=(86400,), dtype=[('req_ts', '<i8'), ('exch_ts', '<i8'), ('resp_ts', '<i8'), ('_padding', '<i8')])

In [7]:
df_order_latency = pl.DataFrame(order_latency)
df_order_latency

req_ts,exch_ts,resp_ts,_padding
i64,i64,i64,i64
1772409600954881550,1772409600974407750,1772409600989052400,0
1772409601989106362,1772409602077531810,1772409602143850896,0
1772409602961586180,1772409602979930900,1772409602993689440,0
1772409603976628721,1772409603995143605,1772409604009029768,0
1772409604955901785,1772409604975508925,1772409604990214280,0
…,…,…,…
1772495995988617135,1772495996003085675,1772495996013937080,0
1772495996986552580,1772495997068762900,1772495997130420640,0
1772495997988565647,1772495998070828235,1772495998132525176,0


Checks if latency has invalid negative values.

检查延迟是否出现无效的负值。

In [8]:
order_entry_latency = df_order_latency['exch_ts'] - df_order_latency['req_ts']
order_resp_latency = df_order_latency['resp_ts'] - df_order_latency['exch_ts']

In [9]:
(order_entry_latency <= 0).sum()

0

In [10]:
(order_resp_latency <= 0).sum()

0

Here, we wrap the entire process into a method with `njit` for increased speed.

在这里，我们将整个过程封装成一个使用 `njit` 的方法，以提高运行速度。

In [11]:
import numpy as np
from numba import njit
import polars as pl
from hftbacktest import LOCAL_EVENT, EXCH_EVENT

@njit
def generate_order_latency_nb(data, order_latency, mul_entry, offset_entry, mul_resp, offset_resp):
    for i in range(len(data)):
        exch_ts = data[i].exch_ts
        local_ts = data[i].local_ts
        feed_latency = local_ts - exch_ts
        order_entry_latency = mul_entry * feed_latency + offset_entry
        order_resp_latency = mul_resp * feed_latency + offset_resp

        req_ts = local_ts
        order_exch_ts = req_ts + order_entry_latency
        resp_ts = order_exch_ts + order_resp_latency

        order_latency[i].req_ts = req_ts
        order_latency[i].exch_ts = order_exch_ts
        order_latency[i].resp_ts = resp_ts

def generate_order_latency(feed_file, output_file = None, mul_entry = 1, offset_entry = 0, mul_resp = 1, offset_resp = 0):
    data = np.load(feed_file)['data']
    df = pl.DataFrame(data)
    
    df = df.filter(
        (pl.col('ev') & EXCH_EVENT == EXCH_EVENT) & (pl.col('ev') & LOCAL_EVENT == LOCAL_EVENT)
    ).with_columns(
        pl.col('local_ts').alias('ts')
    ).group_by_dynamic(
        'ts', every='1000000000i'
    ).agg(
        pl.col('exch_ts').last(),
        pl.col('local_ts').last()
    ).drop('ts')
    
    data = df.to_numpy(structured=True)

    order_latency = np.zeros(len(data), dtype=[('req_ts', 'i8'), ('exch_ts', 'i8'), ('resp_ts', 'i8'), ('_padding', 'i8')])
    generate_order_latency_nb(data, order_latency, mul_entry, offset_entry, mul_resp, offset_resp)

    if output_file is not None:
        np.savez_compressed(output_file, data=order_latency)

    return order_latency

In [12]:
order_latency = generate_order_latency('usdm/ethusdc_20260302.npz', output_file='usdm/feed_latency_ethusdc_20260302.npz', mul_entry=4, mul_resp=3)